# 02 — Dynamic Recommendation Layer

**Builds on:** `01_core_scoring_engine.ipynb` (Phase 1)

**Addresses:** Hao et al. (2025) *Beyond the Hotspots*, Section 6.3 — *Limitations*

> *"A significant limitation of the current framework is the treatment of tourists as a homogeneous group. Future work should distinguish between international and domestic visitor segments, incorporate thematic preference alignment, and account for group size heterogeneity when modelling destination suitability."*
> — Hao et al. (2025), §6.3

---

### What this notebook adds

| Phase 1 Limitation | Phase 2 Fix |
|---|---|
| Static attractiveness weights (all tourists treated equally) | Dynamic weights by `tourist_type` (`international` vs `native`) |
| Static feasibility penalties (all groups treated equally) | Dynamic penalties scaled by `group_size` |
| No thematic alignment between anchor and orbit | Jaccard similarity hard filter on destination `tags` |
| Congestion-only relevance signal | Thematic relevance multiplier gates the final score |

---

### Pipeline Overview

```
User Parameters
 └─ anchor_region, group_size, tourist_type
        │
        ▼
   [Content Filter]          tags → Jaccard similarity_score
        │
        ▼
   [Dynamic Attractiveness]  weights shift by tourist_type
        │
        ▼
   [Dynamic Penalty]         thresholds shift by group_size
        │
        ▼
   [Phase 1 Congestion]      load_score + delay_score → congestion_index
        │
        ▼
   dynamic_match_score = (attractiveness × similarity) − (congestion × 0.5) − penalty
```

---
## Step 1 — Dataset Schema Update

Two new columns extend the Phase 1 schema:

- **`cleanliness_score`** *(0–100)*: Operationalises the *Service* and *Experience* sentiment dimensions identified through ABSA in Hao et al. (2025). In the paper, these were the most polarising dimensions in negative reviews — poor cleanliness and facility quality were the dominant complaints at overtouristed sites.

- **`tags`** *(set of strings)*: Enables content-based thematic filtering. A tourist choosing Kyoto for cherry blossom temples should not be redirected to an alpine hiking village — tags enforce this constraint via Jaccard similarity in Step 3.

In [2]:
import pandas as pd
import numpy as np

# ── Extended dataset (Phase 1 + cleanliness_score + tags) ─────────────────
data = [
    {
        "region": "Kyoto",
        "is_anchor": True,
        "tourist_count": 120000,
        "capacity": 80000,
        "avg_delay_minutes": 30,
        "cultural_score": 95,
        "experiential_score": 90,
        "infrastructure_score": 85,
        "promotion_score": 95,
        "cleanliness_score": 72,   # Degraded by overtourism pressure
        "has_coach_parking": True,
        "group_dining_nearby": True,
        "requires_long_walk": False,
        "tags": {"cherry_blossom", "temple", "historic"}
    },
    {
        "region": "Tokyo",
        "is_anchor": True,
        "tourist_count": 200000,
        "capacity": 150000,
        "avg_delay_minutes": 25,
        "cultural_score": 85,
        "experiential_score": 95,
        "infrastructure_score": 95,
        "promotion_score": 100,
        "cleanliness_score": 80,
        "has_coach_parking": True,
        "group_dining_nearby": True,
        "requires_long_walk": False,
        "tags": {"urban", "modern", "shopping", "food"}
    },
    {
        "region": "Hiroshima",
        "is_anchor": True,
        "tourist_count": 90000,
        "capacity": 70000,
        "avg_delay_minutes": 20,
        "cultural_score": 88,
        "experiential_score": 80,
        "infrastructure_score": 75,
        "promotion_score": 85,
        "cleanliness_score": 78,
        "has_coach_parking": True,
        "group_dining_nearby": True,
        "requires_long_walk": False,
        "tags": {"historic", "memorial", "peace", "temple"}
    },
    {
        "region": "Kanazawa",
        "is_anchor": False,
        "tourist_count": 25000,
        "capacity": 50000,
        "avg_delay_minutes": 8,
        "cultural_score": 85,
        "experiential_score": 80,
        "infrastructure_score": 70,
        "promotion_score": 60,
        "cleanliness_score": 88,   # Undervisited — facilities in good condition
        "has_coach_parking": True,
        "group_dining_nearby": True,
        "requires_long_walk": False,
        "tags": {"cherry_blossom", "historic", "garden"}
    },
    {
        "region": "Takayama",
        "is_anchor": False,
        "tourist_count": 18000,
        "capacity": 40000,
        "avg_delay_minutes": 6,
        "cultural_score": 80,
        "experiential_score": 75,
        "infrastructure_score": 65,
        "promotion_score": 55,
        "cleanliness_score": 90,
        "has_coach_parking": False,
        "group_dining_nearby": True,
        "requires_long_walk": True,
        "tags": {"alpine", "mountain", "rural"}
    },
    {
        "region": "Tottori",
        "is_anchor": False,
        "tourist_count": 12000,
        "capacity": 35000,
        "avg_delay_minutes": 4,
        "cultural_score": 70,
        "experiential_score": 78,
        "infrastructure_score": 60,
        "promotion_score": 40,
        "cleanliness_score": 85,
        "has_coach_parking": True,
        "group_dining_nearby": False,
        "requires_long_walk": False,
        "tags": {"rural", "nature", "sand_dunes"}
    },
    {
        "region": "Akita",
        "is_anchor": False,
        "tourist_count": 9000,
        "capacity": 30000,
        "avg_delay_minutes": 3,
        "cultural_score": 65,
        "experiential_score": 72,
        "infrastructure_score": 55,
        "promotion_score": 35,
        "cleanliness_score": 82,
        "has_coach_parking": False,
        "group_dining_nearby": False,
        "requires_long_walk": True,
        "tags": {"rural", "nature", "festival", "mountain"}
    },
    {
        "region": "Shimane",
        "is_anchor": False,
        "tourist_count": 7000,
        "capacity": 25000,
        "avg_delay_minutes": 2,
        "cultural_score": 75,
        "experiential_score": 70,
        "infrastructure_score": 50,
        "promotion_score": 30,
        "cleanliness_score": 87,
        "has_coach_parking": True,
        "group_dining_nearby": False,
        "requires_long_walk": False,
        "tags": {"historic", "shrine", "garden", "rural"}
    },
]

df = pd.DataFrame(data)

# Preview schema
df[[
    "region", "is_anchor", "cleanliness_score", "tags"
]]

,region,is_anchor,cleanliness_score,tags
0,Kyoto,True,72,"{historic, temple, cherry_blossom}"
1,Tokyo,True,80,"{shopping, urban, modern, food}"
2,Hiroshima,True,78,"{memorial, temple, peace, historic}"
3,Kanazawa,False,88,"{historic, cherry_blossom, garden}"
4,Takayama,False,90,"{mountain, rural, alpine}"
5,Tottori,False,85,"{rural, nature, sand_dunes}"
6,Akita,False,82,"{festival, rural, nature, mountain}"
7,Shimane,False,87,"{rural, shrine, historic, garden}"


---
## Step 2 — Content-Based Filtering via Jaccard Similarity

Jaccard similarity measures thematic overlap between two destinations:

$$J(A, B) = \frac{|A \cap B|}{|A \cup B|}$$

- **Score = 1.0** → identical tag sets (perfect thematic match)
- **Score = 0.0** → no shared tags → orbit is **hard-filtered out** (match score forced to 0)

This directly addresses the Hao et al. (2025) §6.3 critique: a system that recommends Takayama (alpine/mountain) to a Kyoto cherry-blossom visitor is not solving overtourism — it is generating irrelevant diversion.

In [3]:
# ── Jaccard similarity ─────────────────────────────────────────────────────
def calculate_jaccard_similarity(orbit_tags: set, anchor_tags: set) -> float:
    """
    Returns the Jaccard similarity between two tag sets.

    J(A, B) = |A ∩ B| / |A ∪ B|

    Returns 0.0 if either set is empty (no union possible).
    """
    if not orbit_tags or not anchor_tags:
        return 0.0
    intersection = len(orbit_tags & anchor_tags)
    union        = len(orbit_tags | anchor_tags)
    return intersection / union


# ── Sanity checks ──────────────────────────────────────────────────────────
kyoto_tags    = {"cherry_blossom", "temple", "historic"}
kanazawa_tags = {"cherry_blossom", "historic", "garden"}
takayama_tags = {"alpine", "mountain", "rural"}

print(f"Kyoto ↔ Kanazawa : {calculate_jaccard_similarity(kanazawa_tags, kyoto_tags):.4f}")
print(f"Kyoto ↔ Takayama : {calculate_jaccard_similarity(takayama_tags, kyoto_tags):.4f}")
print(f"Kyoto ↔ Kyoto    : {calculate_jaccard_similarity(kyoto_tags, kyoto_tags):.4f}")

Kyoto ↔ Kanazawa : 0.5000
Kyoto ↔ Takayama : 0.0000
Kyoto ↔ Kyoto    : 1.0000


---
## Step 3 — Dynamic Scoring Function

### 3A — Thematic Relevance (Hard Filter)
Jaccard similarity is computed between every candidate orbit and the selected anchor. A score of 0.0 collapses the final match score to 0, suppressing that destination from recommendations regardless of how low its congestion is.

### 3B — Dynamic Feasibility Penalty
Group size gates which operational constraints matter:

| Constraint | Group ≥ 15 (Coach tour) | Group < 15 (FIT / small group) |
|---|---|---|
| No coach parking | −20 | 0 (irrelevant) |
| No group dining | −15 | 0 (irrelevant) |
| Requires long walk | −10 | −5 (minor inconvenience) |

### 3C — Dynamic Attractiveness Weights
Directly fixes the §6.3 homogeneity limitation:

| Dimension | International | Native |
|---|---|---|
| `cultural_score` | **0.40** | 0.10 |
| `cleanliness_score` | 0.30 | 0.20 |
| `experiential_score` | 0.20 | **0.40** |
| `infrastructure_score` | 0.10 | **0.30** |

International visitors prioritise cultural novelty and cleanliness (consistent with Asahi & Habu 2025 — public transport and sanitation dominated their disappointment reasons). Native visitors, already familiar with cultural context, weight experiential richness and infrastructure reliability more heavily.

### 3D — Final Dynamic Match Score
$$\text{dynamic\_match\_score} = (\text{attractiveness} \times \text{similarity\_score}) - (\text{congestion\_index} \times 0.5) - \text{dynamic\_penalty}$$

Multiplying attractiveness by similarity (rather than subtracting it) means a partial thematic match *scales* the attractiveness signal proportionally, rather than applying a flat penalty.

In [4]:
# ── Phase 1 congestion helpers (reproduced for self-contained execution) ───
def piecewise_load_score(ratio: float) -> float:
    """Piecewise congestion score with no ceiling above ratio >= 1.2."""
    if ratio < 0.3:
        return ratio * (20 / 0.3)
    elif ratio < 0.7:
        return 20 + (ratio - 0.3) * (30 / 0.4)
    elif ratio < 0.9:
        return 50 + (ratio - 0.7) * (25 / 0.2)
    elif ratio < 1.0:
        return 75 + (ratio - 0.9) * (15 / 0.1)
    elif ratio < 1.2:
        return 90 + (ratio - 1.0) * (10 / 0.2)
    else:
        return 100 + ((ratio - 1.2) * 20)


def compute_congestion_index(load_score: float, delay_score: float) -> float:
    """Dominance Rule: delay amplifies when load_score >= 100."""
    if load_score >= 100:
        return load_score + (delay_score * 0.20)
    return (load_score * 0.70) + (delay_score * 0.30)


def congestion_flag(index: float) -> str:
    if index < 20:   return "Severely undertouristed"
    elif index < 40: return "Undertouristed"
    elif index < 75: return "Balanced"
    elif index < 100: return "Approaching overtourism"
    else:             return "Severely overtouristed"


# ── Main dynamic recommendation function ──────────────────────────────────
def generate_recommendations(
    df: pd.DataFrame,
    anchor_region: str,
    group_size: int,
    tourist_type: str,
) -> pd.DataFrame:
    """
    Generate dynamic ranked recommendations for orbit destinations.

    Parameters
    ----------
    df            : Extended Phase 1 DataFrame (must include `tags` and
                    `cleanliness_score` columns).
    anchor_region : Name of the overtouristed source destination.
    group_size    : Number of people in the travel group.
    tourist_type  : 'international' or 'native'.

    Returns
    -------
    DataFrame of orbit destinations sorted by dynamic_match_score DESC.
    """

    tourist_type = tourist_type.lower().strip()
    if tourist_type not in ("international", "native"):
        raise ValueError("tourist_type must be 'international' or 'native'")

    # ── Retrieve anchor ────────────────────────────────────────────────────
    anchor_row = df.loc[df["region"] == anchor_region]
    if anchor_row.empty:
        raise ValueError(f"Anchor region '{anchor_region}' not found in dataset.")
    anchor_tags = anchor_row["tags"].values[0]

    # ── Work on orbits only (exclude the anchor itself) ────────────────────
    orbits = df[df["region"] != anchor_region].copy()

    results = []

    for _, row in orbits.iterrows():

        # ── 3A: Jaccard similarity (thematic hard filter) ──────────────────
        similarity_score = calculate_jaccard_similarity(row["tags"], anchor_tags)

        # ── 3B: Dynamic feasibility penalty ───────────────────────────────
        if group_size >= 15:
            dynamic_penalty = (
                (0 if row["has_coach_parking"]  else 20) +
                (0 if row["group_dining_nearby"] else 15) +
                (10 if row["requires_long_walk"] else 0)
            )
        else:
            dynamic_penalty = (5 if row["requires_long_walk"] else 0)

        # ── 3C: Dynamic attractiveness weights ────────────────────────────
        if tourist_type == "international":
            attractiveness = (
                row["cultural_score"]        * 0.40 +
                row["cleanliness_score"]      * 0.30 +
                row["experiential_score"]     * 0.20 +
                row["infrastructure_score"]   * 0.10
            )
        else:  # native
            attractiveness = (
                row["experiential_score"]     * 0.40 +
                row["infrastructure_score"]   * 0.30 +
                row["cleanliness_score"]       * 0.20 +
                row["cultural_score"]          * 0.10
            )

        # ── 3D: Phase 1 congestion index ───────────────────────────────────
        load_ratio  = row["tourist_count"] / row["capacity"]
        load_score  = piecewise_load_score(load_ratio)
        delay_score = min((row["avg_delay_minutes"] / 45) * 100, 100)
        c_index     = compute_congestion_index(load_score, delay_score)
        c_flag      = congestion_flag(c_index)

        # ── 3D: Final dynamic match score ──────────────────────────────────
        # similarity_score == 0 collapses score to 0 via multiplication
        raw_score = (
            (attractiveness * similarity_score)
            - (c_index * 0.5)
            - dynamic_penalty
        )
        dynamic_match_score = round(raw_score, 2)

        results.append({
            "region":               row["region"],
            "is_anchor":            row["is_anchor"],
            "tags":                 row["tags"],
            "similarity_score":     round(similarity_score, 4),
            "attractiveness":       round(attractiveness, 2),
            "congestion_index":     round(c_index, 2),
            "congestion_flag":      c_flag,
            "dynamic_penalty":      dynamic_penalty,
            "dynamic_match_score":  dynamic_match_score,
        })

    output = (
        pd.DataFrame(results)
        .sort_values("dynamic_match_score", ascending=False)
        .reset_index(drop=True)
    )

    return output


print("Functions loaded successfully.")

Functions loaded successfully.


---
## Step 4 — Scenario Execution

### Scenario 1: Large International Coach Tour
- **Anchor:** Kyoto
- **Group size:** 35 (coach tour — full feasibility penalties apply)
- **Tourist type:** International

**Expected behaviour:** Cultural weight dominates attractiveness. Coach parking and dining penalties are active. Thematically unrelated destinations (Takayama alpine, Tottori/Akita rural-nature) are suppressed by the Jaccard filter.

In [5]:
scenario_1 = generate_recommendations(
    df           = df,
    anchor_region= "Kyoto",
    group_size   = 35,
    tourist_type = "international",
)

print("Scenario 1 — Anchor: Kyoto | Group: 35 | Type: International")
print("=" * 65)

DISPLAY = [
    "region", "similarity_score", "attractiveness",
    "congestion_flag", "dynamic_penalty", "dynamic_match_score"
]

scenario_1[DISPLAY].style \
    .format({
        "similarity_score":    "{:.4f}",
        "attractiveness":      "{:.2f}",
        "dynamic_match_score": "{:.2f}",
    }) \
    .background_gradient(subset=["dynamic_match_score"], cmap="RdYlGn") \
    .background_gradient(subset=["similarity_score"], cmap="Blues") \
    .map(
        lambda v: "color: crimson; font-weight: bold"
            if v == "Severely overtouristed"
        else ("color: darkorange" if v == "Approaching overtourism" else ""),
        subset=["congestion_flag"]
    ) \
    .set_caption(
        "Scenario 1 — Kyoto | Coach Tour (n=35) | International"
    )

Scenario 1 — Anchor: Kyoto | Group: 35 | Type: International


,region,similarity_score,attractiveness,congestion_flag,dynamic_penalty,dynamic_match_score
0,Kanazawa,0.5000,83.40,Undertouristed,0,26.78
1,Shimane,0.1667,75.10,Severely undertouristed,15,-9.68
2,Hiroshima,0.4000,82.10,Severely overtouristed,0,-22.46
3,Tottori,0.0000,75.10,Severely undertouristed,15,-24.46
4,Takayama,0.0000,80.50,Undertouristed,30,-42.94
5,Akita,0.0000,70.50,Severely undertouristed,45,-53.00
6,Tokyo,0.0000,86.50,Severely overtouristed,0,-56.89


### Scenario 2: Small Native FIT Pair
- **Anchor:** Kyoto
- **Group size:** 2 (FIT — coach parking and dining penalties waived)
- **Tourist type:** Native

**Expected behaviour:** Experiential and infrastructure weights now dominate. Feasibility penalties are minimal (long-walk penalty only, halved to −5). The ranking order should shift relative to Scenario 1, as destinations with high experiential richness (and solid infrastructure) gain relative to culturally-weighted ones.

In [6]:
scenario_2 = generate_recommendations(
    df           = df,
    anchor_region= "Kyoto",
    group_size   = 2,
    tourist_type = "native",
)

print("Scenario 2 — Anchor: Kyoto | Group: 2 | Type: Native")
print("=" * 65)

scenario_2[DISPLAY].style \
    .format({
        "similarity_score":    "{:.4f}",
        "attractiveness":      "{:.2f}",
        "dynamic_match_score": "{:.2f}",
    }) \
    .background_gradient(subset=["dynamic_match_score"], cmap="RdYlGn") \
    .background_gradient(subset=["similarity_score"], cmap="Blues") \
    .map(
        lambda v: "color: crimson; font-weight: bold"
            if v == "Severely overtouristed"
        else ("color: darkorange" if v == "Approaching overtourism" else ""),
        subset=["congestion_flag"]
    ) \
    .set_caption(
        "Scenario 2 — Kyoto | FIT Pair (n=2) | Native"
    )

Scenario 2 — Anchor: Kyoto | Group: 2 | Type: Native


,region,similarity_score,attractiveness,congestion_flag,dynamic_penalty,dynamic_match_score
0,Kanazawa,0.5000,79.10,Undertouristed,0,24.63
1,Shimane,0.1667,67.90,Severely undertouristed,0,4.12
2,Tottori,0.0000,73.20,Severely undertouristed,0,-9.46
3,Akita,0.0000,68.20,Severely undertouristed,5,-13.00
4,Takayama,0.0000,75.50,Undertouristed,5,-17.94
5,Hiroshima,0.4000,78.90,Severely overtouristed,0,-23.74
6,Tokyo,0.0000,91.00,Severely overtouristed,0,-56.89


---
## Scenario Comparison — Side-by-Side Rank Shift

Merging both outputs lets us directly observe how the same destinations rank differently under different user parameters — the core claim of the dynamic layer.

In [7]:
# ── Side-by-side rank comparison ───────────────────────────────────────────
s1_ranks = (
    scenario_1[["region", "dynamic_match_score"]]
    .rename(columns={"dynamic_match_score": "score_S1_international_n35"})
    .assign(rank_S1=lambda x: x["score_S1_international_n35"].rank(ascending=False).astype(int))
)

s2_ranks = (
    scenario_2[["region", "dynamic_match_score"]]
    .rename(columns={"dynamic_match_score": "score_S2_native_n2"})
    .assign(rank_S2=lambda x: x["score_S2_native_n2"].rank(ascending=False).astype(int))
)

comparison = s1_ranks.merge(s2_ranks, on="region")
comparison["rank_shift"] = comparison["rank_S1"] - comparison["rank_S2"]

comparison = comparison.sort_values("rank_S1").reset_index(drop=True)

def style_shift(v):
    if v > 0:  return "color: green; font-weight: bold"   # rose in S2
    if v < 0:  return "color: crimson; font-weight: bold" # fell in S2
    return ""

comparison.style \
    .format({
        "score_S1_international_n35": "{:.2f}",
        "score_S2_native_n2":         "{:.2f}",
    }) \
    .map(style_shift, subset=["rank_shift"]) \
    .set_caption(
        "Rank Shift: S1 (International, n=35) → S2 (Native, n=2) "
        "| Positive = rose in native ranking"
    )

,region,score_S1_international_n35,rank_S1,score_S2_native_n2,rank_S2,rank_shift
0,Kanazawa,26.78,1,24.63,1,0
1,Shimane,-9.68,2,4.12,2,0
2,Hiroshima,-22.46,3,-23.74,6,-3
3,Tottori,-24.46,4,-9.46,3,1
4,Takayama,-42.94,5,-17.94,5,0
5,Akita,-53.00,6,-13.00,4,2
6,Tokyo,-56.89,7,-56.89,7,0


---
## Addressing §6.3 — Tourist Group Heterogeneity

### How Dynamic Parameterisation Fixes the Hao et al. (2025) Limitations

Hao et al. (2025) identified three specific weaknesses in overtourism diversion frameworks that treat visitors as a homogeneous mass:

---

**Limitation 1: Tourists treated as a homogeneous group**

The Phase 1 attractiveness score applied identical weights to all visitors. An international visitor drawn to Kyoto for its cultural heritage and a domestic visitor seeking experiential novelty would receive identical recommendations — despite having fundamentally different motivations.

**Fix (Step 3C):** The `tourist_type` parameter shifts attractiveness weights dynamically. For `international` tourists, `cultural_score` carries 40% weight and `cleanliness_score` 30% — consistent with the finding in Asahi & Habu (2025) that public transport issues and facility quality were the dominant complaint categories among inbound visitors. For `native` tourists, `experiential_score` leads at 40% and `infrastructure_score` at 30%, reflecting that domestic visitors, already culturally familiar with the destination context, evaluate trips primarily on experiential richness and practical convenience.

---

**Limitation 2: Failure to account for group size and operational context**

A solo FIT traveller and a 40-person coach tour face categorically different logistical constraints. Penalising a boutique rural destination for lacking coach parking when the query is a pair of independent travellers produces incorrect recommendations.

**Fix (Step 3B):** The `group_size` parameter gates which penalties apply. Groups of 15 or more activate the full Phase 1 penalty stack (coach parking −20, group dining −15, long walk −10). For groups below 15, coach parking and dining penalties are waived entirely, and the long walk penalty is halved to −5, treating it as a minor inconvenience rather than an operational blocker.

---

**Limitation 3: No thematic alignment between anchor and recommended orbit**

A congestion-only model will recommend any low-traffic destination, regardless of whether it shares the thematic profile that attracted the tourist to the anchor in the first place. Diverting a visitor from Kyoto temples to Tottori sand dunes does not solve overtourism — it generates visitor dissatisfaction and fails to distribute demand within the relevant thematic corridor.

**Fix (Step 3A):** Jaccard similarity computes tag-set overlap between anchor and orbit. The similarity score acts as a **multiplicative gate** on attractiveness — a destination with zero shared tags produces a match score of 0, regardless of how low its congestion is. Partial overlap proportionally scales the attractiveness contribution, ensuring that near-matches (e.g. Kanazawa sharing `cherry_blossom` and `historic` with Kyoto) are rewarded, while thematically orthogonal destinations are suppressed.

---

### Summary Formula

$$\text{dynamic\_match\_score} = (\underbrace{\text{attractiveness}(\theta_{\text{type}})}_\text{visitor-type weighted} \times \underbrace{J(\text{orbit, anchor})}_\text{thematic filter}) - (\underbrace{\text{congestion\_index}}_\text{Phase 1} \times 0.5) - \underbrace{\text{penalty}(\text{group\_size})}_\text{size-scaled}$$

Each term corresponds directly to one of the three §6.3 limitations. The system no longer assumes a universal tourist — it models the intersection of *who is travelling*, *how many are travelling*, and *why they chose the anchor in the first place*.

In [8]:
import ipywidgets as widgets
from IPython.display import display, clear_output

# ── Output widget (contains the rendered DataFrame) ───────────────────────
out = widgets.Output()

# ── Control widgets ────────────────────────────────────────────────────────
anchor_dropdown = widgets.Dropdown(
    options=sorted(df["region"].unique().tolist()),
    value="Kyoto",
    description="Anchor:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="280px"),
)

group_slider = widgets.IntSlider(
    value=10,
    min=1,
    max=50,
    step=1,
    description="Group Size:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="400px"),
)

type_dropdown = widgets.Dropdown(
    options=["international", "native"],
    value="international",
    description="Tourist Type:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="280px"),
)

# ── Styled output renderer ─────────────────────────────────────────────────
def render_recommendations(anchor_region, group_size, tourist_type):
    results = generate_recommendations(
        df           = df,
        anchor_region= anchor_region,
        group_size   = group_size,
        tourist_type = tourist_type,
    )

    DISPLAY = [
        "region", "similarity_score", "attractiveness",
        "congestion_flag", "dynamic_penalty", "dynamic_match_score",
    ]

    styled = (
        results[DISPLAY].style
        .format({
            "similarity_score":    "{:.4f}",
            "attractiveness":      "{:.2f}",
            "dynamic_match_score": "{:.2f}",
        })
        .background_gradient(subset=["dynamic_match_score"], cmap="RdYlGn")
        .background_gradient(subset=["similarity_score"],    cmap="Blues")
        .map(
            lambda v: "color: crimson; font-weight: bold"
                if v == "Severely overtouristed"
            else ("color: darkorange; font-weight: bold"
                if v == "Approaching overtourism" else ""),
            subset=["congestion_flag"],
        )
        .set_caption(
            f"DSS Recommendations — Anchor: {anchor_region} | "
            f"Group: {group_size} | Type: {tourist_type.capitalize()}"
        )
    )

    with out:
        clear_output(wait=True)
        display(styled)

# ── Wire widgets to renderer ───────────────────────────────────────────────
def _on_change(_):
    render_recommendations(
        anchor_dropdown.value,
        group_slider.value,
        type_dropdown.value,
    )

anchor_dropdown.observe(_on_change, names="value")
group_slider.observe(_on_change,    names="value")
type_dropdown.observe(_on_change,   names="value")

# ── Layout and display ─────────────────────────────────────────────────────
controls = widgets.VBox([
    widgets.HTML("<h3 style='margin-bottom:8px'>⛩ Tourism DSS — Interactive Parameter Panel</h3>"),
    widgets.HBox([anchor_dropdown, type_dropdown]),
    group_slider,
    widgets.HTML(
        "<small style='color:grey'>Group ≥ 15 activates full coach-tour penalties. "
        "Destinations with 0 tag overlap are suppressed regardless of congestion.</small>"
    ),
])

display(controls, out)

# Trigger initial render
render_recommendations(
    anchor_dropdown.value,
    group_slider.value,
    type_dropdown.value,
)

Output()